# Demand forecast: train, validate, forecast

Reads `forecast_sales_history` (written by the `forecast-prepare` job), learns the daily unit sales of
every store and product family with a gradient-boosting model, checks it against a naive baseline on
the last weeks of history, then forecasts the days whose `units` are still empty and writes them to
`forecast_predictions`.

Run as a graal **notebook job**: the cell tagged `parameters` can be overridden by the job's
`parameters`, the executed notebook is kept as the run's output, and the metrics are tracked in the
project's MLflow experiment.

In [ ]:
# Parameters (overridden by the job's `parameters`)
validation_days = 56
max_iter = 300
learning_rate = 0.05
max_leaf_nodes = 31
experiment = "demand-forecast"

In [ ]:
import os

import numpy as np
import pandas as pd
import pyarrow as pa
from sklearn.ensemble import HistGradientBoostingRegressor

import graal  # the tenant's Iceberg catalog, provided to notebook runs

data = graal.read_table("forecast_sales_history")
data["day"] = pd.to_datetime(data["day"])
data = data.sort_values(["store", "family", "day"]).reset_index(drop=True)
known = data["units"].notna()
print(f"{len(data)} rows, {known.sum()} with sales, {(~known).sum()} to forecast")
print(f"sales from {data.loc[known, 'day'].min().date()} to {data.loc[known, 'day'].max().date()}")

## Features

Every feature is known at least 28 days in advance, so one model forecasts the whole horizon at once:
sales of the same series 28, 35, 42 and 364 days earlier, the mean of days 28 to 55 before, the
calendar (weekday, season), and the promotions and closures, planned ahead.

In [ ]:
series = data.groupby(["store", "family"])["units"]
for lag in (28, 35, 42, 364):
    data[f"lag_{lag}"] = series.shift(lag)
data["mean_28_55"] = series.transform(lambda s: s.shift(28).rolling(28, min_periods=14).mean())
data["weekday"] = data["day"].dt.day_of_week
angle = 2 * np.pi * data["day"].dt.day_of_year / 365.25
data["season_sin"], data["season_cos"] = np.sin(angle), np.cos(angle)
data["store_code"] = data["store"].astype("category").cat.codes
data["family_code"] = data["family"].astype("category").cat.codes
data["promotion"] = data["promotion"].astype(int)
data["closed"] = data["closed"].astype(int)

FEATURES = [
    "lag_28",
    "lag_35",
    "lag_42",
    "lag_364",
    "mean_28_55",
    "weekday",
    "season_sin",
    "season_cos",
    "store_code",
    "family_code",
    "promotion",
    "closed",
]
CATEGORICAL = [FEATURES.index(c) for c in ("weekday", "store_code", "family_code")]
usable = known & data["lag_28"].notna()
print(f"{usable.sum()} training rows, {len(FEATURES)} features")

## Validation on the last weeks of history

In [ ]:
last_day = data.loc[known, "day"].max()
cut = last_day - pd.Timedelta(days=validation_days)
train = data[usable & (data["day"] <= cut)]
valid = data[usable & (data["day"] > cut)]


def model():
    return HistGradientBoostingRegressor(
        max_iter=max_iter,
        learning_rate=learning_rate,
        max_leaf_nodes=max_leaf_nodes,
        categorical_features=CATEGORICAL,
        random_state=0,
    )


def wape(actual, predicted):
    return float(np.abs(actual - predicted).sum() / actual.sum())


candidate = model().fit(train[FEATURES], train["units"].astype(float))
predicted = np.clip(candidate.predict(valid[FEATURES]), 0, None) * (1 - valid["closed"])
actual = valid["units"].astype(float)
metrics = {
    "wape": wape(actual, predicted),
    "wape_baseline_lag_28": wape(actual, valid["lag_28"].astype(float)),
    "mae": float(np.abs(actual - predicted).mean()),
}
metrics["improvement_vs_baseline"] = 1 - metrics["wape"] / metrics["wape_baseline_lag_28"]
for name, value in metrics.items():
    print(f"{name:>26}: {value:.4f}")
assert metrics["wape"] < metrics["wape_baseline_lag_28"], (
    "the model does not beat the naive baseline"
)

## Final model and forecast

In [ ]:
final = model().fit(data.loc[usable, FEATURES], data.loc[usable, "units"].astype(float))
future = data[~known].copy()
future["predicted_units"] = (
    np.clip(final.predict(future[FEATURES]), 0, None) * (1 - future["closed"])
).round(1)

run_id = None
if os.environ.get("MLFLOW_TRACKING_URI"):  # set by graal when the project's MLflow tracking is on
    import mlflow

    mlflow.set_experiment(experiment)
    with mlflow.start_run(run_name="gradient-boosting") as run:
        mlflow.log_params(
            {
                "validation_days": validation_days,
                "max_iter": max_iter,
                "learning_rate": learning_rate,
                "max_leaf_nodes": max_leaf_nodes,
                "features": ",".join(FEATURES),
                "training_rows": int(usable.sum()),
            }
        )
        mlflow.log_metrics(metrics)
        run_id = run.info.run_id
    print(f"MLflow run {run_id} in experiment {experiment!r}")

predictions = future[["store", "family", "day", "predicted_units"]].assign(
    day=future["day"].dt.date, model_run=run_id or "untracked"
)
graal.write_table(
    pa.Table.from_pandas(predictions, preserve_index=False).cast(
        pa.schema(
            [
                ("store", pa.string()),
                ("family", pa.string()),
                ("day", pa.date32()),
                ("predicted_units", pa.float64()),
                ("model_run", pa.string()),
            ]
        )
    ),
    "forecast_predictions",
    mode="overwrite",
)
print(
    f"forecast_predictions: {len(predictions)} rows, "
    f"{predictions['day'].min()} to {predictions['day'].max()}"
)
predictions.groupby("family")["predicted_units"].sum().round(0)